In [3]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD MERGED DATASET
# ============================================================

# The error 'ValueError: Worksheet named 'Merged_dataset' not found' means the specified sheet does not exist in the Excel file.
# Let's list the actual sheet names in the file to help identify the correct one.
try:
    excel_file = pd.ExcelFile("/content/merged.xlsx")
    print(f"Available sheets in '/content/merged.xlsx': {excel_file.sheet_names}")
    # If you see a different sheet name (e.g., ['Sheet1']), update the 'sheet_name' argument below.
    # For example: sheet_name='Sheet1'
except FileNotFoundError:
    print("Error: '/content/merged.xlsx' not found. Please ensure the file is correctly uploaded.")
except Exception as e:
    print(f"An error occurred while trying to list sheet names: {e}")

# IMPORTANT: Update the 'sheet_name' argument below with one of the names printed above.
# For example, if the output was ['DataSheet'], change "Merged_dataset" to "DataSheet".
df = pd.read_excel(
    "/content/merged.xlsx",
    sheet_name="Sheet1" # <--- This is the argument you need to update
)

print("Dataset loaded successfully!")
print("Original Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 2. BASIC CLEANING
# ============================================================

# Remove duplicate research-quality column if it exists
if "the_research_quality.1" in df.columns:
    df.drop(columns=["the_research_quality.1"], inplace=True)

# Remove rows where University Name is missing
df.dropna(subset=["University_Name"], inplace=True)

# Remove duplicate universities
df.drop_duplicates(
    subset=["University_Name"],
    inplace=True
)

# Reset index
df.reset_index(drop=True, inplace=True)


# ============================================================
# 3. CONVERT REQUIRED COLUMNS TO NUMERIC
# ============================================================

numeric_columns = [
    "Rank",
    "Academic_Reputation_SCORE",
    "Faculty_Student_Ratio_SCORE",
    "Citations_per_Faculty_SCORE",
    "International_Student_SCORE",
    "the_research_env",
    "the_research_quality"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# ============================================================
# 4. GLOBAL RANKING SCORE
# ============================================================

# Better rank = higher score

max_rank = df["Rank"].max()

df["Global_Ranking_Score"] = (
    (max_rank - df["Rank"] + 1) / max_rank
) * 100


# ============================================================
# 5. RESEARCH IMPACT SCORE
# ============================================================

df["Research_Impact_Score"] = (
    df["Citations_per_Faculty_SCORE"]
)


# ============================================================
# 6. FACULTY-TO-STUDENT RATIO
# ============================================================

# Your dataset has the QS Faculty Student Ratio score.
# There are no separate Faculty and Student count columns.

df["Faculty_to_Student_Ratio"] = (
    df["Faculty_Student_Ratio_SCORE"]
)


# ============================================================
# 7. INTERNATIONAL STUDENT PERCENTAGE
# ============================================================

# Your dataset provides International Student Score,
# not raw international-student and total-student counts.

df["International_Student_Percentage"] = (
    df["International_Student_SCORE"]
)


# ============================================================
# 8. ACADEMIC REPUTATION SCORE
# ============================================================

df["Academic_Reputation_Score"] = (
    df["Academic_Reputation_SCORE"]
)


# ============================================================
# 9. RESEARCH PRODUCTIVITY INDEX
# ============================================================

# Use available research-related indicators

research_columns = [
    "Citations_per_Faculty_SCORE",
    "the_research_env",
    "the_research_quality"
]

df["Research_Productivity_Index"] = (
    df[research_columns].mean(
        axis=1,
        skipna=True
    )
)


# ============================================================
# 10. DISPLAY KPI RESULTS
# ============================================================

kpi_columns = [
    "Global_Ranking_Score",
    "Research_Impact_Score",
    "Faculty_to_Student_Ratio",
    "International_Student_Percentage",
    "Academic_Reputation_Score",
    "Research_Productivity_Index"
]

print("\nKPI Results:")
display(
    df[
        [
            "University_Name",
            "Country"
        ] + kpi_columns
    ].head(10)
)


# ============================================================
# 11. CHECK KPI SUMMARY
# ============================================================

print("\nKPI Summary:")
display(df[kpi_columns].describe())


# ============================================================
# 12. SAVE FINAL DATASET
# ============================================================

df.to_excel(
    "university_final_dataset.xlsx",
    index=False
)

print("\n======================================")
print("KPI ENGINEERING COMPLETED SUCCESSFULLY")
print("======================================")
print("Final Shape:", df.shape)
print("File saved as: university_final_dataset.xlsx")

Available sheets in '/content/merged.xlsx': ['Sheet1']
Dataset loaded successfully!
Original Shape: (1504, 18)

First 5 rows:


,University_Name,Rank,Country,Academic_Reputation_SCORE,Employer_Reputation_SCORE,Faculty_Student_Ratio_SCORE,Citations_per_Faculty_SCORE,International_Faculty _SCORE,International_Student_SCORE,International_Students_Diversity_SCORE,International_Research_Network_SCORE,Employment_Outcomes_SCORE,Sustainability_SCORE,the_teaching,the_research_env,the_research_quality,the_intl_outlook,Score_Final
0,massachusetts institute of technology (mit),1,United States of America,100.0,100.0,100.0,100.0,100.0,91.6,92.3,94.1,100.0,93.8,91.5,99.8,97.2,74.2,100
1,imperial college london,2,United States of America,99.6,100.0,99.3,95.0,100.0,100.0,100.0,97.5,95.9,98.3,83.4,92.1,87.4,55.2,99.4
2,stanford university,3,United Kingdom,100.0,100.0,100.0,99.7,94.2,73.5,76.1,96.5,100.0,95.4,95.2,99.6,99.4,88.4,98.9
3,university of oxford,4,United States of America,100.0,100.0,100.0,91.0,98.8,98.6,98.7,100.0,100.0,77.9,97.8,98.4,95.8,66.4,97.9
4,harvard university,5,United Kingdom,100.0,100.0,98.3,100.0,79.1,81.4,60.6,99.4,100.0,77.8,95.4,99.2,97.4,77.4,97.7



KPI Results:


,University_Name,Country,Global_Ranking_Score,Research_Impact_Score,Faculty_to_Student_Ratio,International_Student_Percentage,Academic_Reputation_Score,Research_Productivity_Index
0,massachusetts institute of technology (mit),United States of America,100.000000,100.0,100.0,91.6,100.0,99.000000
1,imperial college london,United States of America,99.928622,95.0,99.3,100.0,99.6,91.500000
2,stanford university,United Kingdom,99.857245,99.7,100.0,73.5,100.0,99.566667
3,university of oxford,United States of America,99.785867,91.0,100.0,98.6,100.0,95.066667
4,harvard university,United Kingdom,99.714490,100.0,98.3,81.4,100.0,98.866667
5,university of cambridge,United States of America,99.643112,88.6,100.0,93.1,100.0,94.533333
6,eth zurich (swiss federal institute of technol...,United Kingdom,99.571734,98.8,71.7,99.3,99.7,90.800000
7,national university of singapore (nus),Switzerland,99.500357,95.9,71.5,96.9,99.9,87.166667
8,ucl (university college london),Singapore,99.428979,80.9,94.8,100.0,99.9,83.366667
9,california institute of technology (caltech),United Kingdom,99.357602,100.0,100.0,90.7,98.3,98.866667



KPI Summary:


,Global_Ranking_Score,Research_Impact_Score,Faculty_to_Student_Ratio,International_Student_Percentage,Academic_Reputation_Score,Research_Productivity_Index
count,800.000000,1504.000000,1504.000000,1504.000000,1504.000000,1504.000000
mean,66.061206,30.476862,33.988497,33.334628,25.753391,30.244441
std,27.788114,29.717867,28.444370,32.360335,24.487338,29.265569
min,0.071378,1.000000,1.000000,1.000000,1.000000,1.000000
25%,57.458958,6.000000,10.800000,6.075000,8.775000,6.000000
50%,71.484654,18.050000,23.600000,20.600000,16.000000,18.050000
75%,85.760171,50.025000,50.525000,55.650000,32.550000,50.200000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000



KPI ENGINEERING COMPLETED SUCCESSFULLY
Final Shape: (1504, 24)
File saved as: university_final_dataset.xlsx
